# Post-process 6 hour iSnobal outputs into 4 timesteps per day, each consisting of 6 hours

In [ ]:
# Current Timestep breakdown for example Day 91
# 5 total timesteps per day, unequally distributed:
# hour 00 = SWI flux value at hour 00 to save, computed since end of previous day's hour 23 (covers 1 hour)
# hour 06 = flux values summed over hours 01–06 (6 hours)
# hour 12 = flux summed over hours 07–12 (6 hours)
# hour 18 = flux summed over hours 13–18 (6 hours)
# hour 23 = flux summed over hours 19–23 (5 hours)


# Idealized timestep breakdown for example Day 91
# h06 (01-06, 6 hours)
# h12 (07-12, 6 hours)
# h18 (13-18, 6 hours)
# "new h24"(19-23 + h 00 from Day 92, 6 hours)

### Test with Animas outputs

In [ ]:
import os
import sys

import numpy as np
import xarray as xr
import pandas as pd

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')

In [ ]:
# Set up directories
model_dir = '/uufs/chpc.utah.edu/common/home/skiles-group2/model_runs_jmh'

# Build a dict using thisvar as keys
var_dict = {
    'SWI': {
        'filename': 'em',
        'dropvarlist': ['net_rad', 'sensible_heat', 'latent_heat', 'snow_soil', 'precip_advected', 'sum_EB',
                        'evaporation', 'snowmelt', 'cold_content', 'projection']
    },
    'specific_mass': {
        'filename': 'snow',
        'dropvarlist': ['thickness', 'snow_density', 'liquid_water', 'temp_surf', 'temp_lower', 'temp_snowcover', 'thickness_lower', 'water_saturation', 'projection']
    }
}

In [ ]:
def pretty_time(ds, hours_only=False):
    '''Convert time values of input dataset to 'YYYY-MM-DD HH' format strings.'''
    if hours_only:
        return [f'h{np.datetime_as_string(t.values, unit="h")[-2:]}' for t in ds.time]
    else:
        return ds.time.values.astype('M8[h]').astype(str)

def check_sums(ds, data_var='SWI', daily_only=False, return_sums=False):
    """
    Compute array sum for each timestep and then daily sum
    """
    pretty_hours = pretty_time(ds[data_var], hours_only=True)
    if not daily_only:
        for jdx, t in enumerate(ds[data_var].time):
            # Print just the hours, removing YYYY-MM-DD part and
            # Sum at this timestep
            print(f'{pretty_hours[jdx]}: {ds[data_var].sel(time=t).sum().values:.0f}')
        # Print the daily total
    else:
        daily_total = ds[data_var].sum(dim='time').sum().values
        print(f"Daily total: {daily_total:.0f}")
    print('---')
    if return_sums:
        return daily_total

def combine_hour_23_and_00(ds, data_var='SWI', debug=False):
    """
    Combines hour 23 and hour 00 for input variable, replaces hour 00 with sum, removes hour 23.
    """
    # Get hours
    hours = ds.time.dt.hour
    if debug:
        print(hours.values)
    # Sum variable data for hour 23 and hour 00
    value_23 = ds[data_var].sel(time=hours == 23).squeeze('time')
    value_00 = ds[data_var].sel(time=hours == 0).squeeze('time')
    combined = value_23 + value_00
    if debug:
        print(f"Hour 23 value: {value_23.values}")
        print(f"Hour 00 value: {value_00.values}")
        print(f"Combined value: {combined.values}")
    # Create modified dataset
    if debug:
        print("Creating modified dataset, removing hour 23...")
    ds_modified = ds.sel(time=hours != 23)  # Remove hour 23
    if debug:
        print("...and assigning combined value to hour 00")
        print(ds_modified.time[ds_modified.time.dt.hour == 0])
    ds_modified[data_var].loc[dict(time=ds_modified.time[ds_modified.time.dt.hour == 0])] = combined

    return ds_modified

def remove_hour_23(ds):
    """
    Removes hour 23 from the dataset for the specified variable.
    """
    hours = ds.time.dt.hour
    ds_modified = ds.sel(time=hours != 23)  # Remove hour 23
    return ds_modified

def process_year_state(ds, data_var='specific_mass', output_dir="daily_output"):
    """
    Remove hour 23 for each day and save as separate daily netCDF files.
    Midnight (hour 00) is grouped with the previous day.
    """
    # Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Shift time by 1 hour so midnight belongs to previous day
    shifted_time = ds.time - pd.Timedelta(hours=1)

    # Group by date using shifted time
    daily_groups = ds.groupby(shifted_time.dt.date)
    processed_days = []
    skipped_days = []

    for date, day_data in daily_groups:
        try:
            # Remove hour 23
            processed_day = remove_hour_23(day_data, data_var=data_var)

            # Save to netCDF
            filename = f"{output_dir}/{data_var}_{date.strftime('%Y%m%d')}.nc"
            processed_day.to_netcdf(filename)
            print(f"Saved: {filename}")

            processed_days.append(date)

        except Exception as e:
            print(f"Error processing {date}: {e}")
            skipped_days.append(date)
    return processed_days, skipped_days

def process_year_flux(ds, data_var='SWI', output_dir="daily_output", debug=False):
    """
    Apply hour 23/00 combination for each day and save as separate netCDF files.
    Midnight (hour 00) is grouped with the previous day.
    """
    # Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Shift time by 1 hour so midnight belongs to previous day
    shifted_time = ds.time - pd.Timedelta(hours=1)

    # Group by date using shifted time
    daily_groups = ds.groupby(shifted_time.dt.date)
    mismatch_days = []
    processed_days = []
    skipped_days = []

    for date, day_data in daily_groups:
        try:
            # Only process if day has both hour 23 and hour 00
            hours = day_data.time.dt.hour
            if debug:
                print(hours)
            if (23 in hours.values) and (0 in hours.values):
                before = check_sums(day_data, data_var=data_var, daily_only=True, return_sums=True)
                if debug:
                    print("Attempting processing")
                processed_day = combine_hour_23_and_00(day_data, data_var=data_var, debug=False)
                if debug:
                    print("Processing complete")
                after = check_sums(processed_day, data_var=data_var, daily_only=True, return_sums=True)
                if before != after:
                    print(f"Warning: Daily sum mismatch on {date}: before={before}, after={after}")
                    mismatch_days.append(date)
            else:
                # Keep day unchanged if it doesn't have both hours
                processed_day = day_data
            # Save to netCDF
            filename = f"{output_dir}/{data_var}_{date.strftime('%Y%m%d')}.nc"
            processed_day.to_netcdf(filename)
            print(f"Saved: {filename}")

            processed_days.append(date)

        except Exception as e:
            print(f"Error processing {date}: {e}")
            skipped_days.append(date)
    return mismatch_days, processed_days, skipped_days

In [ ]:
# # Load the files into xarray datasets
# model_dir = '/uufs/chpc.utah.edu/common/home/skiles-group2/model_runs_jmh'

# # First assess SWI outputs from all the tests
# em_fn_list = h.fn_list(model_dir, 'animas*/wy2021/*100m/*20201101/em.nc')
# print(em_fn_list)

# em_ds_list = [xr.open_dataset(f, drop_variables=dropvarlist) for f in em_fn_list]

# # Compute SWI for each timestep and sum the total for the day
# em_ds_list[0]['SWI'].time

# for jdx, ds in enumerate(em_ds_list):
#     print(PurePath(em_fn_list[jdx].split('/wy2021')[0]).stem)
#     # Loop through each timestep
#     for t in ds['SWI'].time:
#         # Print just the hours,  removing YYYY-MM-DD part
#         print(f'h{np.datetime_as_string(t.values, unit="h")[-2:]}', end='')
#         # Sum SWI at this timestep
#         print(f': {ds['SWI'].sel(time=t).sum().values:.0f}')
#     # Print the daily SWI total
#     print(f"Daily total: {ds['SWI'].sum(dim='time').sum().values:.0f}")
#     print('---')


# Takeaway for RTI reproductions → use mt25 and HRRR_CBR (which is what you shared) to re-generate all 6 hour outputs

In [ ]:
# # Post processing tests, use isnobal_rti
# # Test with WY 2021
# # Read in all the em.nc [SWI] files from model runs
# wy = 2021
# basin = 'animas'
# rti_dir = h.fn_list(model_dir, f'{basin}*rti/wy{wy}/*100m/')[0]
# em_ds = xr.open_mfdataset(h.fn_list(rti_dir, '*/em.nc'), drop_variables=dropvarlist)

# # Section out the xarray dataset into daily chunks with midnight lumped into the previous day
# em_subset = em_ds.isel(time=slice(31*5, 31*5+5))  # Day 31
# pretty_time(em_subset)
# check_sums(em_subset)
# check_sums(em_subset, daily_only=True)

In [ ]:
# new_ds = combine_hour_23_and_00(em_subset)
# check_sums(new_ds)

In [ ]:
# Usage
basin = 'animas'
wy = 2021

# Specify variable of interest and which variables to drop
thisvar = 'SWI' #'specific_mass'
filename = var_dict[thisvar]['filename']
dropvarlist = var_dict[thisvar]['dropvarlist']
print(basin, wy, thisvar)
print(filename, dropvarlist)
rti_dir = h.fn_list(model_dir, f'{basin}*rti/wy{wy}/*100m/')[0]
ds = xr.open_mfdataset(h.fn_list(rti_dir, f'*/{filename}.nc'), drop_variables=dropvarlist, parallel=True)
if thisvar == 'SWI':
    mismatch_days, processed_days, skipped_days = process_year_flux(ds, data_var=thisvar, output_dir=f"./{basin}_wy{wy}_{thisvar}")
else:
    processed_days, skipped_days = process_year_state(ds, data_var=thisvar, output_dir=f"./{basin}_wy{wy}")

# sub_ds = ds.isel(time=slice(31*5, 31*5+5))
# pretty_time(sub_ds)
# mismatch_days, processed_days, skipped_days = process_year_flux(sub_ds, data_var=thisvar, output_dir=f"./{basin}_wy{wy}_{thisvar}", debug=True)

In [ ]:
# # Usage
# for wy in 2023, 2024:
#     print(wy)
#     rti_dir = h.fn_list(model_dir, f'{basin}*rti/wy{wy}/*100m/')[0]
#     print(rti_dir)
#     ds = xr.open_mfdataset(h.fn_list(rti_dir, f'*/{filename}.nc'), drop_variables=dropvarlist, parallel=True)
#     processed_days, skipped_days = process_year_state(ds, data_var=thisvar, output_dir=f"./{basin}_wy{wy}")

In [ ]:
# Select specific days to assess processing
# if mismatch days do not exist, choose a random selection of 7 days
if len(mismatch_days) == 0:
    print("No mismatch days found, selecting random days for assessment.")
    all_days = pd.date_range(start=f'{wy-1}-10-01', end=f'{wy}-09-30', freq='D')
    test_days = sorted(np.random.choice(all_days, size=7, replace=False))
else:
    test_days = mismatch_days

test_days = [pd.to_datetime(f'{wy}-09-30')]
for test_day in test_days:
    print(f"Assessing day: {test_day}")
    day_str = test_day.strftime('%Y%m%d')
    # Locate the original file based on the day_str
    original_fn = h.fn_list(rti_dir, f'run{day_str}/em.nc')[0]

    # And also grab the following day
    next_day_str = (test_day + pd.Timedelta(days=1)).strftime('%Y%m%d')
    next_day_fn = h.fn_list(rti_dir, f'run{next_day_str}/em.nc')[0]
    processed_fn = f'./animas_wy{wy}_SWI/SWI_{day_str}.nc'

    original_ds = xr.open_dataset(original_fn, drop_variables=dropvarlist)
    next_day_ds = xr.open_dataset(next_day_fn, drop_variables=dropvarlist)
    processed_ds = xr.open_dataset(processed_fn)

    # Section out the specific day from the original dataset
    shifted_time = original_ds.time - pd.Timedelta(hours=1)
    day_mask = shifted_time.dt.date == test_day
    original_day_ds = original_ds.sel(time=day_mask)
    # Note the next day hour 00
    next_day_hour_00 = next_day_ds.sel(time=next_day_ds.time.dt.hour == 0)

    print("Original Day Sums:")
    check_sums(original_day_ds)
    check_sums(next_day_hour_00)
    # Print the combination of original day hour 23 and + next day hour 00
    combined_value = (original_day_ds['SWI'].sel(time=original_day_ds.time.dt.hour == 23).squeeze('time') +
                      next_day_hour_00['SWI'].squeeze('time'))
    print(f'Combined h23 + next day h00: {combined_value.sum().values:.0f}')

    print("Processed Day Sums:")
    check_sums(processed_ds)

    print("==============================")